# Solar-MACH
**Multi-spacecraft longitudinal configuration plotter**

- GitHub: https://github.com/jgieseler/solarmach
- PyPI: https://pypi.org/project/solarmach
- conda: https://anaconda.org/conda-forge/solarmach
- web app: https://solar-mach.github.io
---

## Local installation (not needed here)

In order to run it locally on your computer (that is, not on [pyhcschool.heliocloud.org](https://pyhcschool.heliocloud.org)), you need to install `solarmach` via pip:
``` bash
$ pip install solarmach
```

or via conda:
``` bash
$ conda install -c conda-forge solarmach
```
---

## Importing 

In [ ]:
from solarmach import SolarMACH, print_body_list  # print_body_list only needed to get a list of available bodies/spacecraft

%config InlineBackend.figure_formats = ['svg']  # edit notebook plotting setting (optional)

---
## 1. Minimal example

Necessary options are a list of wanted spacecraft/bodies, the corresponding solar wind speeds (in km/s), and the date of interest:

In [ ]:
body_list = ['Earth', 'Solar Orbiter', 'PSP']
vsw_list = [400, 400, 400]             # position-sensitive solar wind speed per body in body_list
date = '2022-6-1 12:00:00'

Initialize the SolarMACH object for these options:

In [ ]:
sm1 = SolarMACH(date, body_list, vsw_list)

And produce the final plot:

In [ ]:
sm1.plot(plot_sun_body_line=True, show_earth_centered_coord=False)

---

## 2. Example with all the details

First, get a list of available bodies/spacecraft:

In [ ]:
print(print_body_list().index)

Provide the necessary options, this time for more spacecraft:

In [ ]:
body_list = ['Mercury', 'Venus', 'Earth', 'Mars', 'STEREO A', 'STEREO B', 'Solar Orbiter', 'PSP', 'BepiColombo']
vsw_list = len(body_list) * [350]        # position-sensitive solar wind speed per body in body_list
date = '2021-6-1 12:00:00'

Now we also want to indicate the position (in [Carrington coordinates](https://docs.sunpy.org/en/stable/generated/api/sunpy.coordinates.frames.HeliographicCarrington.html)) and direction of a flare, and the (assumed) solar wind speed at its location:

In [ ]:
reference_long = 0                               # Carrington longitude of reference (None to omit)
reference_lat = 0                                # Carrington latitude of reference (None to omit)
reference_vsw = 400                              # define solar wind speed at reference

In addition, we explicitly provide all availabe plotting options:

In [ ]:
plot_spirals = True                              # plot Parker spirals for each body
plot_sun_body_line = False                       # plot straight line between Sun and body
show_earth_centered_coord = False                # display Earth-aligned coordinate system
transparent = False                              # make output figure background transparent
numbered_markers = True                          # plot each body with a numbered marker
filename = f'Solar-MACH_{date.replace(" ", "_")}.png'  # define filename of output figure

Finally, initializing and plotting with these options. If `outfile` is provided, the plot will be saved next to the Notebook with the provided `filename`.

In [ ]:
sm2 = SolarMACH(date, body_list, vsw_list, reference_long, reference_lat)

sm2.plot(plot_spirals=plot_spirals,
         plot_sun_body_line=plot_sun_body_line,
         show_earth_centered_coord=show_earth_centered_coord,
         reference_vsw=reference_vsw,
         transparent=transparent,
         numbered_markers=numbered_markers,
         outfile=filename
         )

All the data can also be obtained as a Pandas DataFrame for further use:

In [ ]:
df = sm2.coord_table
display(df)

In [ ]:
df['Heliocentric distance (AU)'].values

---

## 3. Example using Stonyhurst coordinates for reference

Let's take a look at the situation at the first ground-level enhancement (GLE) of solar cycle 25 on 28 October 2021 

First, we just provide some options as before:

In [ ]:
body_list = ['STEREO-A', 'Earth', 'BepiColombo', 'PSP', 'Solar Orbiter', 'Mars']
vsw_list = [340, 300, 350, 350, 320, 350]        # position-sensitive solar wind speed per body in body_list
date = '2021-10-28 15:20:00'

# optional parameters
plot_spirals = True                              # plot Parker spirals for each body
plot_sun_body_line = True                        # plot straight line between Sun and body
show_earth_centered_coord = True                # display Earth-aligned coordinate system
transparent = False                              # make output figure background transparent
numbered_markers = True                          # plot each body with a numbered marker
filename = f'Solar-MACH_{date.replace(" ", "_")}.png'  # define filename of output figure

But now we want to provide the coordinates of the flare in [Stonyhurst coordinates](https://docs.sunpy.org/en/stable/generated/api/sunpy.coordinates.frames.HeliographicStonyhurst.html) (instead of [Carrington](https://docs.sunpy.org/en/stable/generated/api/sunpy.coordinates.frames.HeliographicCarrington.html)). For this, we have to convert them manually to Carrington coordinates for further use (at least at the moment). Note that this conversion is time-dependent.

In [ ]:
import astropy.units as u
from astropy.coordinates import SkyCoord
from sunpy.coordinates import frames

reference_long = 2                               # Stonyhurst longitude of reference (None to omit)
reference_lat = 26                               # Stonyhurst latitude of reference (None to omit)
coord = SkyCoord(reference_long*u.deg, reference_lat*u.deg, frame=frames.HeliographicStonyhurst, obstime=date)
coord = coord.transform_to(frames.HeliographicCarrington(observer='Sun'))
reference_long = coord.lon.value                 # Carrington longitude of reference
reference_lat = coord.lat.value                  # Carrington latitude of reference

reference_vsw = 300                              # define solar wind speed at reference

Finally, initializing and plotting with these options:

In [ ]:
sm3 = SolarMACH(date, body_list, vsw_list, reference_long, reference_lat)
sm3.plot(plot_spirals=plot_spirals,
         plot_sun_body_line=plot_sun_body_line,
         show_earth_centered_coord=show_earth_centered_coord,
         reference_vsw=reference_vsw,
         transparent=transparent,
         numbered_markers=numbered_markers,
         outfile=filename
         )

---
## 4. Only obtain data as Pandas DataFrame

We can also just obtain a table with the spatial data, without producing a plot at all.

First provide necessary options:

In [ ]:
body_list = ['STEREO-A', 'Earth', 'BepiColombo', 'PSP', 'Solar Orbiter', 'Mars']
vsw_list = [400, 400, 400, 400, 400, 400]        # position-sensitive solar wind speed per body in body_list
date = '2022-6-1 12:00:00'

Then initialize `SolarMACH` and obtain data as Pandas DataFrame:

In [ ]:
sm4 = SolarMACH(date, body_list, vsw_list)
df = sm4.coord_table
display(df)

If we also provide the `reference` information, it will be available in the table, too:

In [ ]:
sm4 = SolarMACH(date, body_list, vsw_list, reference_long=273, reference_lat=7)
df = sm4.coord_table
display(df)

---
# 5. Ideas for further usage

## 5.1 Loop over multiple datetimes (plots)

This might be useful to either:

- read-in an *event catalog*, and loop over those datetimes to quickly get the constellations of all these events,
- create a series of daily constellation plots, and combine them into one animation:

In [ ]:
body_list = ['Mercury', 'Venus', 'Earth', 'Mars', 'STEREO A', 'STEREO B', 'Solar Orbiter', 'PSP', 'BepiColombo']
vsw_list = len(body_list) * [350]                # position-sensitive solar wind speed per body in body_list
plot_spirals = True                              # plot Parker spirals for each body
plot_sun_body_line = False                       # plot straight line between Sun and body
show_earth_centered_coord = True                 # display Earth-aligned coordinate system
transparent = False                              # make output figure background transparent
numbered_markers = True                          # plot each body with a numbered marker

for i in range(1,31,1):    
    j = str(i).rjust(2, '0')
    date = f'2022-6-{j} 12:00:00'
    filename = f'Solar-MACH_{date.replace(" ", "_")}.png'  # define filename of output figure

    sm5 = SolarMACH(date, body_list, vsw_list)
    sm5.plot(plot_spirals=plot_spirals,
             plot_sun_body_line=plot_sun_body_line,
             show_earth_centered_coord=show_earth_centered_coord,
             transparent=transparent,
             numbered_markers=numbered_markers,
             outfile=filename
             )

Get a sorted list of the files just created using `glob`:

In [ ]:
import glob
files = sorted(glob.glob(filename.replace(f'{i}', '*')))

Build an animated GIF out of these files using `imageio`:

In [ ]:
import imageio
with imageio.get_writer('solarmach.gif', mode='I') as writer:
    for filename in files:
        image = imageio.imread(filename)
        writer.append_data(image)

![Animated GIF](solarmach.gif)

## 5.2 Loop over multiple datetimes (only data)

For example, to look for spacecraft alignments, like *"When are PSP and Solar Orbiter at the same magnetic footpoint?"*

In [ ]:
body_list = ['Mercury', 'Venus', 'Earth', 'Mars', 'STEREO A', 'Solar Orbiter', 'PSP', 'BepiColombo']
vsw_list = len(body_list) * [350]                # position-sensitive solar wind speed per body in body_list

df = []
dates = []
for i in range(1,31,1):
    date = f'2022-6-{i} 12:00:00'
    filename = f'Solar-MACH_{date.replace(" ", "_")}.png'  # define filename of output figure

    sm6 = SolarMACH(date, body_list, vsw_list)
    df = df + [sm6.coord_table]
    dates = dates + [date]

In [ ]:
display(df[0])
display(df[1])
display(df[2])

In [ ]:
display(dates)

---

# 6. Let's take a look at the web app!

Just open https://solar-mach.github.io in the web browser